# Hazus flood damage functions

## Hazus vulnerability functions
The [FEMA Hazus](https://www.fema.gov/flood-maps/tools-resources/flood-map-products/hazus/user-technical-manuals) team have provided an extract of the damage functions used in the Hazus 6.1 Flood in an Excel file HazusFloodDamageFunctions_Hazus61.xlsx. This is a highly granular, albeit US-specific, set of damage functions.

There is a point of attention around the use of Hazus flood damage functions. The functions provide damage
for negative flood depths. This is because the functions are based on flood depth *relative to the height
of the first finished floor*. The difference is material in cases where the first finished floor is elevated,
perhaps with a crawl space or basement underneath. The correct way of applying the functions is to calculate
a relative depth, $d_r$ which is the difference between the absolute flood depth $d_a$ and the height
of the first finished floor above grade, $d_f$,

$$d_r = d_a - d_f$$

It can also be important to use a minimum relative absolute depth threshold, $d_t$ in applying the
function. To illustrate this, say the finished floor is 1 m above grade and the absolute flood depth
is 0 m. Some curves have non-zero damage at -4 feet (-1.22 m) flood depth. Since $d_r$ is -1.0 m,
we would see a non-zero damage. To avoid this a threshold, $d_t$ can be added to the absolute
flood depth, below which the damage is considered to be zero. The logic is then:

if $d_a < d_t$ then damage is zero, otherwise damage is $v(d_r)$, $v$ being the Hazus damage function.

It may be desirable to apply this logic in code, as part of the vulnerability model, so that the calculation
can react to asset-specific information. On the other hand, it may be desirable to calculate a function
based on absolute flood depth, which is applied to categories of assets.

For this reason, the config builder has an option ```adjust_to_absolute_depth```. If set to False, 
the curves will be returned as a function of relative depth:
the same convention as Hazus. Otherwise (and the default is True), the curves will be adjusted to be functions of absolute depth
using the ```min_flood_depth``` (default 5 cm) and ```first_floor_height``` (default 30 cm) parameters provided. A limitation is noted that these
parameters are used across all curves, although clearly some Hazus curves are relevant in the case that a
basement is present: in general a more sophisticated approach is needed to adjust _all_ curves.

## Identifying the config lines
For wind, config lines were produced of the form ```hazus_wind_structure_spec_build_type=MLRI,damage_type=structure```, which identifies the structural damage curve for the Hazus Specific Building Type 'MLRI'. For flood these are combined into a single identifier, e.g. ```hazus_fl_structure_dmg_id=800```. The reason for this is that Hazus 6.1 defines different identifiers for structural damage, contents and inventory and the intent is to align with these conventions.

In [ ]:
import plotly.io
from plotly.subplots import make_subplots
from IPython.display import HTML
from physrisk.vulnerability_models.configuration.hazus_config_builders import (
    ConfigBuilderHazusFlood,
)

plotly.io.renderers.default = "notebook"

In [ ]:
# the file is too large to include in the repo; we need to download from S3 before we start as a one-off:
builder = ConfigBuilderHazusFlood()
builder.download_inputs()

In [ ]:
# the building of the configuration items is then done as follows:
builder = ConfigBuilderHazusFlood()
config_items = builder.build_config()
print(f"{len(config_items)} vulnerability configuration items created")
print(config_items[38])

1886 vulnerability configuration items created
hazard_class='CoastalInundation,PluvialInundation,RiverineInundation' asset_class='Asset' asset_identifier='hazus_fl_structure_dmg_id=143' indicator_id='flood_depth/above_first_floor' indicator_units='metres' impact_id='damage' impact_units=None curve_type='indicator/piecewise_linear' points_x=[-1.22, -0.91, -0.61, -0.3, 0.0, 0.3, 0.61, 0.91, 1.22, 1.52, 1.83, 2.13, 2.44, 2.74, 3.05, 3.35, 3.66, 3.96, 4.27, 4.57, 4.88, 5.18, 5.49, 5.79, 6.1, 6.4, 6.71, 7.01, 7.32] points_y=[0.0, 0.0, 0.0, 0.0, 0.01, 0.22, 0.28, 0.31, 0.33, 0.48, 0.48, 0.48, 0.48, 0.56, 0.56, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58, 0.58] points_z=None points_kind=None cap_of_points_x=None cap_of_points_y=None activation_of_points_x=None baseline_quantile_of_points_x=None reference=None license=None


In [ ]:
from physrisk.vulnerability_models.configuration.oed_attribute_matcher import (
    parse_asset_identifier,
)

fig1 = make_subplots(rows=1, cols=1)


def is_low_structure_dmg_id(asset_identifier: str, threshold: int = 120) -> bool:
    # hazus_fl_structure_dmg_id is usually numeric, but utility assets use ids
    # like "util_17"; treat those as not matching rather than raising.
    dmg_id = parse_asset_identifier(asset_identifier).get(
        "hazus_fl_structure_dmg_id", "9990"
    )
    return dmg_id.isdigit() and int(dmg_id) < threshold


config_subset = [
    item for item in config_items if is_low_structure_dmg_id(item.asset_identifier)
]
for item in config_subset:
    fig1.add_scatter(
        x=item.points_x, y=item.points_y, row=1, col=1, name=item.asset_identifier
    )
fig1.update_xaxes(title="Flood depth (m)", title_font={"size": 14}, row=1, col=1)
fig1.update_yaxes(
    title="Damage as fraction of TIV", title_font={"size": 14}, row=1, col=1
)
fig1.update_layout(legend=dict(orientation="h", y=-0.1))
fig1.update_layout(margin=dict(l=20, r=20, t=20, b=20))
fig1.update_layout(
    height=800,
)

# Render explicitly with include_mathjax=False: the default "notebook" renderer
# always embeds its own (legacy v2) MathJax, which conflicts with the MathJax
# already loaded by Sphinx for the $...$ math elsewhere on this page.
HTML(
    fig1.to_html(
        div_id="inundation-hazus-structure-curves",
        include_plotlyjs="cdn",
        include_mathjax=False,
        full_html=False,
    )
)